In [1]:
import matplotlib.pyplot as plt
import numpy as np

import tidy3d as td
import gplugins.tidy3d as gt

nm = 1e-3
wavelength = np.linspace(1500, 1600) * nm
f = td.C_0 / wavelength

In [2]:
nitride_complex = td.material_library["Si3N4"]["Luke2015PMLStable"].eps_model(f)
nitride_index, nitride_k = td.Medium.eps_complex_to_nk(nitride_complex)
box_complex = td.material_library["SiO2"]["Horiba"].eps_model(f)
box_index, box_k = td.Medium.eps_complex_to_nk(box_complex)

In [4]:
from upvfab_design_tools import DC_EME

ModuleNotFoundError: No module named 'skfem'

## 1. Simulating Propagation Modes in SOI Waveguides

In [4]:
# STUDENT! Put your code here.
wavelength = np.linspace(1500,1600,6) * nm  # Student code here. Tip np.linspace()
central = 1550*nm

deep_waveguide = gt.modes.Waveguide(
    wavelength=wavelength, 
    core_width=450 * nm, 
    slab_thickness=0.0,
    core_material='si',
    clad_material='sio2',
    core_thickness=220 * nm,
    num_modes=8,
    cache_path='.cache/',
    precision='double',
    max_grid_scaling=1.2,
    grid_resolution=20, 
)

res_neff = deep_waveguide.n_eff # In this case, the result is not just a number, is a wavelength-dependent vector
res_te = deep_waveguide.fraction_te # Wavelength-dependent vector
res_tm =deep_waveguide.fraction_tm # Wavelength-dependent vector
neff_TE0 = res_neff[:,0].real

coef_TE = np.polyfit(wavelength - central,neff_TE0,2)

n1_TE = coef_TE[2]
n2_TE = coef_TE[1]
n3_TE = coef_TE[0]

ng_TE = n1_TE - central*n2_TE
neff_TM0 = res_neff[:,1].real

coef_TM = np.polyfit(wavelength- central,neff_TM0,2)

n1_TM = coef_TM[2]
n2_TM = coef_TM[1]
n3_TM = coef_TM[0]

ng_TM = n1_TM - central*n2_TM
print(
    f"""
TE0 effective index varies from {neff_TE0[0]:.3f} to {neff_TE0[-1]:.3f}
TM0 effective index varies from {neff_TM0[0]:.3f} to {neff_TM0[-1]:.3f}

ng_TE = {ng_TE:.3f}
ng_TM = {ng_TM:.3f}
"""
)

2026-03-23 10:09:57.541 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\Waveguide_e0ec38b5ceb294b8.npz.

TE0 effective index varies from 2.399 to 2.273
TM0 effective index varies from 1.806 to 1.688

ng_TE = 4.287
ng_TM = 3.571



## 2. Designing Directional Couplers

In [2]:
m = DC_EME() # Here you instantiate a Directional Coupler to be simulated with the Eigen-mode expansion (EME) algorithm.

gap = np.linspace(0.6,1,100)
L=np.linspace(0,8*L_pi,200)

K = []   # coupling coefficient
Lpi = []

for g in gap:

    m.DC_wg_gap = g # Gap between waveguides
    m.DC_wg_width = 1.2 # Width of the waveguides core
    m.DC_N_waveguides = 2

    # 1) Compute de DC modes - Only execute whenever a geometrical parameter is changed. Takes time to find the modes. 
    m.find_all_modes()

    L_pi = m.get_L_pi()
    Lpi.append(L_pi)

    K = np.sin(np.pi/2*(L/L_pi))**2
    K.append(K)

K = np.array(K)
Lpi = np.array(Lpi)

NameError: name 'DC_EME' is not defined